In [1]:
%pip install "pyarrow>=11.0.0"
%pip install pyspark

You should consider upgrading via the '/Users/riya/Instagram_reach/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/riya/Instagram_reach/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession, functions as F

# BUILDING A SPARK SESSION TO CONNECT TO A REMOTE SPARK CLUSTER

import os
os.environ["JAVA_HOME"] = os.popen("/usr/libexec/java_home").read().strip()
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Olist ETL") \
    .getOrCreate()


spark.conf.set("spark.sql.shuffle.partitions", "200")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/16 15:40:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# IMPORTING NECESSARY LIBRARIES

from pyspark.sql.functions import col, countDistinct, year, month, dayofmonth, concat, col, lit, isnan, when, count, desc ,asc, to_date,dayofweek
from pyspark.sql.types import DateType , StructType, StructField
import datetime
from pyspark.sql.window import Window
from pyspark.sql import types as t

In [4]:

orders = spark.read.csv('./Data/olist_orders_dataset.csv', header=True, sep=',', inferSchema=True)
orders.orderBy("order_purchase_Timestamp",ascending=True).show(5)
orders.printSchema()
orders.count()


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|2e7a8482f6fb09756...|08c5351a6aca1c158...|     shipped|     2016-09-04 21:15:19|2016-10-07 13:18:03|         2016-10-18 13:14:51|                         NULL|          2016-10-20 00:00:00|
|e5fa5a7210941f7d5...|683c54fc24d40ee9f...|    canceled|     2016-09-05 00:15:34|2016-10-07 13:17:15|                        NULL|                         NULL|          2016-10-28 00:00:00|
|809a282bbd5dbcabb...|622e13439d6b5a0b4...|  

99441

In [5]:
# Reading the customers dataset

customers_info = spark.read.csv('./Data/olist_customers_dataset.csv', header=True, sep=',', inferSchema=True)
customers_info.show(5)
customers_info.printSchema()
customers_info.select("customer_id").distinct().count()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows
root
 |-- customer_id: string (

99441

In [6]:
# Reading the order items dataset

order_items = spark.read.csv('./Data/olist_order_items_dataset.csv', header=True, sep=',', inferSchema=True)
order_items.show(5)
order_items.dtypes

# Shape of the order_items dataset
print(dict(rows = order_items.count(), columns = len(order_items.columns)))




+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [7]:
# reading the order reviews dataset

order_reviews = spark.read.csv('./Data/olist_order_reviews_dataset.csv', header=True, sep=',', inferSchema=True)
order_reviews.show(5)

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                NULL|                  NULL| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                NULL|                  NULL| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                NULL|  Recebi bem antes ...| 2017-04-21 00:00:00|   

In [8]:

# reading the products dataset

products = spark.read.csv('./Data/olist_products_dataset.csv', header=True, sep=',', inferSchema=True)
products.show(5)
products.printSchema()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [9]:
# reading the sellers dataset

sellers = spark.read.csv('./Data/olist_sellers_dataset.csv', header=True, sep=',', inferSchema=True)
sellers.show(5)
sellers.select("seller_state").distinct().count()

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
+--------------------+----------------------+-----------------+------------+
only showing top 5 rows


23

In [10]:
# reading the order payments dataset

order_payments = spark.read.csv('./Data/olist_order_payments_dataset.csv', header=True, sep=',', inferSchema=True)
order_payments.show(5)
order_payments.printSchema()
# order_payments.select(sum("payment_value")).show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments:

In [11]:
# reading the product category name translation 

product_category_name_translation = spark.read.csv('./Data/product_category_name_translation.csv', header=True, sep=',', inferSchema=True)
product_category_name_translation.show()
product_category_name_translation.printSchema()

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
|        esporte_lazer|               sports_leisure|
|           perfumaria|                    perfumery|
| utilidades_domest...|                   housewares|
|            telefonia|                    telephony|
|   relogios_presentes|                watches_gifts|
|    alimentos_bebidas|                   food_drink|
|                bebes|                         baby|
|            papelaria|                   stationery|
| tablets_impressao...|         tablets_printing_...|
|           brinquedos|                         toys|
|       telefonia_fixa|     

In [12]:

# SHAPE OF THE ORDERS DATASET
print(dict(rows = orders.count(), columns = len(orders.columns)))


{'rows': 99441, 'columns': 8}


In [13]:
from pyspark.sql.functions import col, sum


file_names = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "customers_info": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "product_category_name_translation": "product_category_name_translation.csv"
}

cleaned_dataframes = {}

for name, file in file_names.items():
    print(f"\n=== {name.upper()} ===")
    path = f"./Data/{file}"
    
    # Load
    df = spark.read.option("header", True).csv(path)
    original_count = df.count()

    # Drop duplicates
    df = df.dropDuplicates()
    after_dedup = df.count()

    # Drop rows with nulls
    df = df.dropna()
    final_count = df.count()

    print(f"Original Rows: {original_count}")
    print(f"After Drop Duplicates: {after_dedup}")
    print(f"After Drop Nulls: {final_count}")
    print(f"Dropped: {original_count - final_count}")

    # Optional: show columns with original nulls
    df_nulls = spark.read.option("header", True).csv(path)
    df_nulls.select([
        sum(col(c).isNull().cast("int")).alias(c + "_nulls")
        for c in df_nulls.columns
    ]).show(truncate=False)

    # Save cleaned DataFrame to dictionary
    cleaned_dataframes[name] = df



=== ORDERS ===
Original Rows: 99441
After Drop Duplicates: 99441
After Drop Nulls: 96461
Dropped: 2980
+--------------+-----------------+------------------+------------------------------+-----------------------+----------------------------------+-----------------------------------+-----------------------------------+
|order_id_nulls|customer_id_nulls|order_status_nulls|order_purchase_timestamp_nulls|order_approved_at_nulls|order_delivered_carrier_date_nulls|order_delivered_customer_date_nulls|order_estimated_delivery_date_nulls|
+--------------+-----------------+------------------+------------------------------+-----------------------+----------------------------------+-----------------------------------+-----------------------------------+
|0             |0                |0                 |0                             |160                    |1783                              |2965                               |0                                  |
+--------------+----------------

Original Rows: 1000163
After Drop Duplicates: 738332
After Drop Nulls: 738332
Dropped: 261831
+---------------------------------+---------------------+---------------------+----------------------+-----------------------+
|geolocation_zip_code_prefix_nulls|geolocation_lat_nulls|geolocation_lng_nulls|geolocation_city_nulls|geolocation_state_nulls|
+---------------------------------+---------------------+---------------------+----------------------+-----------------------+
|0                                |0                    |0                    |0                     |0                      |
+---------------------------------+---------------------+---------------------+----------------------+-----------------------+


=== PRODUCT_CATEGORY_NAME_TRANSLATION ===
Original Rows: 71
After Drop Duplicates: 71
After Drop Nulls: 71
Dropped: 0
+---------------------------+-----------------------------------+
|product_category_name_nulls|product_category_name_english_nulls|
+------------------

In [14]:
orders.count()
customers_info.distinct().count()

99441

In [15]:
item_with_cust = order_items.join(
    orders.select("order_id", "customer_id"),
    on="order_id",
    how="inner"
)

# 2. compute LTV as sum(price + freight_value) per customer
ltv_per_customer = (
    item_with_cust
      .groupBy("customer_id")
      .agg(
        F.round(
          F.sum(F.col("price") + F.col("freight_value")),
          2
        ).alias("ltv")
      )
)
# Optionally bring in customer details
ltv_per_customer = ltv_per_customer.join(customers_info, on="customer_id", how="left")

# 3. inspect
ltv_per_customer.orderBy(F.desc("ltv"))\
  .select("customer_id", "ltv")\
  .show(5)


+--------------------+--------+
|         customer_id|     ltv|
+--------------------+--------+
|1617b1357756262bf...|13664.08|
|ec5b2ba62e5743423...| 7274.88|
|c6e2731c5b391845f...| 6929.31|
|f48d464a0baaea338...| 6922.21|
|3fd6777bbce08a352...| 6726.66|
+--------------------+--------+
only showing top 5 rows


In [16]:

# 1. compute avg revenue per order, per customer
avg_rev_per_customer = (
   item_with_cust
      .groupBy("customer_id")
      .agg(
        F.round(
          F.avg(F.col("price") + F.col("freight_value")),
          2
        ).alias("avg_revenue")
      )
      .join(customers_info, on="customer_id", how="left")
)

# 2. inspect the top 20 customers by average revenue
avg_rev_per_customer \
  .orderBy(F.desc("avg_revenue")) \
  .select("customer_id", "customer_city", "customer_state", "avg_revenue") \
  .show(5)


+--------------------+-------------+--------------+-----------+
|         customer_id|customer_city|customer_state|avg_revenue|
+--------------------+-------------+--------------+-----------+
|c6e2731c5b391845f...| campo grande|            MS|    6929.31|
|f48d464a0baaea338...|      vitoria|            ES|    6922.21|
|3fd6777bbce08a352...|      marilia|            SP|    6726.66|
|df55c14d1476a9a34...|     araruama|            RJ|    4950.34|
|24bbf5fd2f2e1b359...|         maua|            SP|    4764.34|
+--------------------+-------------+--------------+-----------+
only showing top 5 rows


In [17]:

from pyspark.sql import SparkSession, functions as F

# 1. Initialize Spark
spark = SparkSession.builder.appName("Olist-Full-Clean-UpperCats").getOrCreate()

# 2. Define file paths for Olist datasets
file_paths = {
    "orders": "./Data/olist_orders_dataset.csv",
    "order_items": "./Data/olist_order_items_dataset.csv",
    "order_payments": "./Data/olist_order_payments_dataset.csv",
    "order_reviews": "./Data/olist_order_reviews_dataset.csv",
    "products": "./Data/olist_products_dataset.csv",
    "sellers": "./Data/olist_sellers_dataset.csv",
    "customers_info": "./Data/olist_customers_dataset.csv",
    "geolocation": "./Data/olist_geolocation_dataset.csv",
    "product_category_name_translation": "./Data/product_category_name_translation.csv"
}


# 3. Define columns for categorical cleaning
cat_cols = ["order_status", "customer_city",
            "product_category_name", "seller_city"]

# 4. Define timestamp columns to split
ts_cols = ["order_purchase_timestamp", "order_approved_at",
           "order_delivered_customer_date", "order_estimated_delivery_date"]


# 5. Clean each file
cleaned = {}

for name, path in file_paths.items():
    print(f"\n=== {name.upper()} ===")
    df = spark.read.option("header", True).csv(path)

 

    # Trim and uppercase relevant columns
    for c in cat_cols:
        if c in df.columns:
            df = df.withColumn(c, F.initcap(F.trim(F.col(c))))


    df = df.dropDuplicates()
    cleaned[name] = df

    # Preview
    df.show(20, truncate=False)
    print("Rows after cleaning:", df.count())



=== ORDERS ===


26/02/16 15:40:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|974c1993ab8024d3ed16229183c2308d|a90391a47de936d56c66a5366cba1462|Delivered   |2017-02-20 11:45:39     |2017-02-22 03:10:20|2017-02-23 06:47:35         |2017-03-09 14:27:58          |2017-03-21 00:00:00          |
|c82430e4a84f03b67cc2a82fecd873f0|a625f7cf467b00ae83700201dce9a6d0|Delivered   |2017-11-03 12:56:49     |2017-11-07 07:30:43|2017-11-07 19:3

In [18]:
from pyspark.sql import functions as F, Window

orders = cleaned["orders"]

# keep ONLY the latest status per order_id  (final business truth)
w = Window.partitionBy("order_id").orderBy(F.col("order_purchase_timestamp").desc())
orders_clean = (orders
                .withColumn("rn", F.row_number().over(w))
                .filter("rn = 1")             # drop duplicates
                .drop("rn"))

# compute distribution once
total = orders_clean.count()
status_pct = (orders_clean.groupBy("order_status")
                         .count()
                         .withColumn("percent", F.round(F.col("count")/total*100, 4)))
status_pct.show(vertical=True, truncate=False)


-RECORD 0-------------------
 order_status | Shipped     
 count        | 1107        
 percent      | 1.1132      
-RECORD 1-------------------
 order_status | Processing  
 count        | 301         
 percent      | 0.3027      
-RECORD 2-------------------
 order_status | Invoiced    
 count        | 314         
 percent      | 0.3158      
-RECORD 3-------------------
 order_status | Created     
 count        | 5           
 percent      | 0.005       
-RECORD 4-------------------
 order_status | Canceled    
 count        | 625         
 percent      | 0.6285      
-RECORD 5-------------------
 order_status | Delivered   
 count        | 96478       
 percent      | 97.0203     
-RECORD 6-------------------
 order_status | Unavailable 
 count        | 609         
 percent      | 0.6124      
-RECORD 7-------------------
 order_status | Approved    
 count        | 2           
 percent      | 0.002       



In [19]:
from pyspark.sql.functions import date_format

# 1. Load cleaned data
orders = cleaned["orders"]
order_items = cleaned["order_items"]

# 2. Rename timestamp columns BEFORE the join
orders = (orders
    .withColumnRenamed("order_purchase_timestamp", "purchase_ts")
    .withColumnRenamed("order_delivered_customer_date", "delivered_ts")
    .withColumnRenamed("order_approved_at", "approved_ts")
    .withColumnRenamed("order_estimated_delivery_date", "estimated_ts")
) 
# # # Extract date and time from renamed timestamp columns

orders1=(orders
    .withColumn("purchase_date", date_format("purchase_ts", "yyyy-MM-dd"))
    .withColumn("purchase_time", date_format("purchase_ts", "HH:mm:ss"))
    .withColumn("approved_date", date_format("approved_ts", "yyyy-MM-dd"))
    .withColumn("approved_time", date_format("approved_ts", "HH:mm:ss"))
    .withColumn("delivered_date", date_format("delivered_ts", "yyyy-MM-dd"))
    .withColumn("delivered_time", date_format("delivered_ts", "HH:mm:ss"))
    .withColumn("estimated_date", date_format("estimated_ts", "yyyy-MM-dd"))
    .withColumn("estimated_time", date_format("estimated_ts", "HH:mm:ss"))
)
# Drop raw timestamp columns 
orders1 = orders1.drop("purchase_ts", "approved_ts", "delivered_ts", "estimated_ts","order_delivered_carrier_date")
orders1.printSchema()
orders1.orderBy("purchase_date", ascending=True).show(5)


orders1.select([sum(col(c).isNull().cast("int")).alias(c) for c in orders1.columns]).show()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- purchase_time: string (nullable = true)
 |-- approved_date: string (nullable = true)
 |-- approved_time: string (nullable = true)
 |-- delivered_date: string (nullable = true)
 |-- delivered_time: string (nullable = true)
 |-- estimated_date: string (nullable = true)
 |-- estimated_time: string (nullable = true)

+--------------------+--------------------+------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+
|            order_id|         customer_id|order_status|purchase_date|purchase_time|approved_date|approved_time|delivered_date|delivered_time|estimated_date|estimated_time|
+--------------------+--------------------+------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+

In [20]:
%pip install "plotly[express]"
%pip install pandas
%pip install --upgrade nbformat

You should consider upgrading via the '/Users/riya/Instagram_reach/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/riya/Instagram_reach/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/riya/Instagram_reach/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [21]:
import plotly.graph_objects as px 

In [22]:

daily_orders = (
    orders1.groupBy("purchase_date").count()
      .orderBy("purchase_date")
)

# 3)  COLLECT SMALL RESULT TO PANDAS -----------------------------
pdf_orders = daily_orders.toPandas()          # only ~1000 rows – safe
import plotly.express as px
# Plotly plot
fig = px.line(pdf_orders, x="purchase_date", y="count", title='Daily Orders - Olist (2016-2018)')
fig.update_layout(
    xaxis_title="purchase_date",
    yaxis_title="purchase_count",
    title_x=0.5,  # Center the title
    title_y=0.95,  # Adjust the vertical position of the title
)

fig.show()

In [23]:
purchase_hour= orders1.withColumn("purchase_hour",F.hour("purchase_time"))\
              .groupBy("purchase_hour").count()\
              .orderBy("purchase_hour")
purchase_hour.show(5)

pdf_orders = purchase_hour.toPandas()  # convert to pandas DataFrame for Plotly

import plotly.express as px
fig = px.bar(pdf_orders, x="purchase_hour", y="count", title = 'Daily Orders - Olist (2016-2018)')
fig.update_layout(
    xaxis_title="Purchase_Hour",
    yaxis_title="Number of hourly orders",
    height=500,
    width=700,
    font=dict(  # Correctly specify the font property
        family="Arial, sans-serif",  # Font family
        size=14,                     # Font size
        color="black"                # Font color
    )
)

fig.update_traces(text=pdf_orders["count"], textposition="inside")  # Display text on bars

fig.show()
          

+-------------+-----+
|purchase_hour|count|
+-------------+-----+
|            0| 2394|
|            1| 1170|
|            2|  510|
|            3|  272|
|            4|  206|
+-------------+-----+
only showing top 5 rows


In [24]:
order_items = cleaned["order_items"]
order_payments = cleaned["order_payments"]
order_items.select('price','freight_value').describe().show()

# Cast to numeric FIRST
order_items = (order_items
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
)

# Create new columns
order_items = (order_items
    .withColumn("gross_value", F.round(F.col("price") + F.col("freight_value"), 2))
    .withColumn("shipping_date", date_format("shipping_limit_date", "yyyy-MM-dd"))
    .withColumn("shipping_time", date_format("shipping_limit_date", "HH:mm:ss"))
    .orderBy("shipping_date", ascending=True)
    .drop("shipping_limit_date")
)
order_items.show(5)
order_items.printSchema()

import pyspark.pandas as ps
import plotly.express as px
from pyspark.sql import functions as F

# 1. Join tables & convert to pandas DataFrame
# Cast payment_value to double before converting to pandas
spark_pdf = (
    order_items
    .join(order_payments.withColumn("payment_value", F.col("payment_value").cast("double")), on="order_id", how="left")
    .select("order_id", "price", "payment_value")
    .filter(F.col("price") < 10000)
    .toPandas()
)

# 2. Aggregate by price and reset index
agg = (
    spark_pdf
    .groupby("price")
    .agg({
        "order_id": "count",
        "payment_value": "sum"
    })
    .rename(columns={
        "order_id": "count_orders",
        "payment_value": "total_revenue"
    })
    .reset_index()
)

# 3. Scatter plot – with valid x, y, size
fig = px.scatter(
    agg,
    x="count_orders",
    y="price",
    size="total_revenue",
    hover_data=["price", "count_orders", "total_revenue"],
    title="Number of Orders by Price (Sized by Revenue)"
)

fig.update_xaxes(
    title="Number of Orders",
    tickformat=",.0f"
)

fig.update_yaxes(
    title="Price (BRL)",
    tickprefix="$",
    tickformat=",.0f"
)

fig.update_layout(
    width=700,
    height=500,
    title_x=0.5,
    font=dict(
        family="Arial, sans-serif",
        size=14,
        color="black"
    )
)

fig.show()

+-------+------------------+------------------+
|summary|             price|     freight_value|
+-------+------------------+------------------+
|  count|            112650|            112650|
|   mean|120.65373901464221|19.990319928983876|
| stddev| 183.6339280502593|15.806405412297105|
|    min|              0.85|              0.00|
|    max|            999.99|             99.97|
+-------+------------------+------------------+

+--------------------+-------------+--------------------+--------------------+-----+-------------+-----------+-------------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|price|freight_value|gross_value|shipping_date|shipping_time|
+--------------------+-------------+--------------------+--------------------+-----+-------------+-----------+-------------+-------------+
|bfbd0f9bdef843021...|            2|5a6b04657a4c5ee34...|ecccfa2bb93b34a3b...|44.99|         2.83|      47.82|   2016-09-19|     23:11:33|
|bfbd0f9bde

/Users/riya/Instagram_reach/venv/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning:

'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.



In [25]:
from pyspark.sql import SparkSession, functions as F

# Access cleaned sellers DataFrame
sellers = cleaned["sellers"]

#  Top 5 ZIP codes with most sellers
sellers.groupBy("seller_zip_code_prefix") \
       .count() \
       .orderBy(F.desc("count")) \
       .show(5, truncate=False)

sellers.printSchema()

+----------------------+-----+
|seller_zip_code_prefix|count|
+----------------------+-----+
|14940                 |49   |
|13660                 |10   |
|16200                 |9    |
|13920                 |9    |
|14020                 |8    |
+----------------------+-----+
only showing top 5 rows
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [26]:
# Top 5 states with most sellers
df_sellers = sellers.groupBy("seller_state").count().orderBy(F.desc("count"))
df_sellers.withColumn("percent",(F.col("count")/sellers.count())*100).show(5)

df1_sellers = df_sellers.toPandas()

import plotly.graph_objects as go         
fig = go.Figure(
    data=[
        go.Pie( labels = df1_sellers["seller_state"], 
                               values = df1_sellers["count"], 
                               hole = 0.5 ,
                               pull = [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])
    ]
)
fig.update_layout(
    title = 'Olist sellers over of all brazillian states (2016-2018)',
    autosize = False,
    width = 700,
    height = 500)
fig.update_traces(textposition = 'inside')


fig.show()
          


+------------+-----+------------------+
|seller_state|count|           percent|
+------------+-----+------------------+
|          SP| 1849|59.741518578352185|
|          PR|  349|11.276252019386106|
|          MG|  244| 7.883683360258481|
|          SC|  190| 6.138933764135703|
|          RJ|  171| 5.525040387722132|
+------------+-----+------------------+
only showing top 5 rows


In [27]:
# Group by state and city and count sellers
grouped_sellers = sellers.groupBy("seller_state", "seller_city").count().orderBy(F.desc("count"))
# Add percentage column (each city-state pair's share of total sellers)
grouped_sellers.withColumn("percent", (F.col("count") / sellers.count()) * 100).show(10, truncate=False)

df1_sellers = grouped_sellers.toPandas()


fig = px.bar(
    df1_sellers,
    x="seller_city",
    y="count",
    title="Seller distribution – Olist (2016-2018)",
)

fig.update_layout(
    xaxis_title="City",
    yaxis_title="Number of sellers",
    title_x=0.5,                       # centre the title
    bargap=0.25,                       # space between bars (optional)
)
fig.update_traces(marker_color='royalblue')  # Change bar color

fig.show()            


+------------+--------------+-----+------------------+
|seller_state|seller_city   |count|percent           |
+------------+--------------+-----+------------------+
|SP          |Sao Paulo     |694  |22.423263327948302|
|PR          |Curitiba      |124  |4.006462035541195 |
|RJ          |Rio De Janeiro|93   |3.0048465266558964|
|MG          |Belo Horizonte|66   |2.132471728594507 |
|SP          |Ribeirao Preto|52   |1.680129240710824 |
|SP          |Guarulhos     |50   |1.615508885298869 |
|SP          |Ibitinga      |49   |1.5831987075928917|
|SP          |Santo Andre   |45   |1.4539579967689822|
|SP          |Campinas      |41   |1.3247172859450727|
|PR          |Maringa       |40   |1.2924071082390953|
+------------+--------------+-----+------------------+
only showing top 10 rows


In [28]:

products = cleaned["products"]


# join, rename, and UPPER-case the English category — all in one chain
products = (
    cleaned["products"]
      .join(cleaned["product_category_name_translation"], "product_category_name", "left")
      .withColumnRenamed("product_category_name_english", "product_category_en")   # remove the stray space!
      .withColumn("product_category_en",
                  F.upper(F.trim("product_category_en")))  # make it UPPER + strip pads
      .drop("product_category_name")  # drop the original            
)
# fill null with median
for col_name in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
     products = products.withColumn(col_name, F.col(col_name).cast("int"))  #cast to int
     

products.show(5) 
products.printSchema()


+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-------------------+
|          product_id|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_category_en|
+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-------------------+
|e7cad5e8ed9b92e78...|                 56|                       663|                 4|             150|               16|                5|              11|     SPORTS_LEISURE|
|e8f22df5fa5b45490...|                 57|                       404|                 2|            1500|               22|               12|              33|      FASHION_SHOES|
|76df076d70248fa44...|                 52|                       219|                 1|            1000|

In [29]:
from pyspark.sql import SparkSession, functions as F,window


# -------------------------
# DESCRIPTIVE STATS
# -------------------------
products.select('product_weight_g','product_length_cm','product_height_cm','product_width_cm').describe().show()

# Top Selling Products
df_products= products.groupBy("product_category_en").count().orderBy(F.desc("count"))

w = Window.orderBy(F.desc("count")).partitionBy(F.lit(1))
df_products.withColumn("percent", (F.col("count") / products.count()) * 100)\
           .withColumn("cum_percent", F.round(F.sum("count").over(w)/F.lit(products.count())*100, 2)).show(5)


df1_products = df_products.toPandas()



fig = px.bar(
    df1_products,
    x="product_category_en",
    y="count",
    title="Top Selling Products – Olist (2016-2018)",
)

fig.update_layout(
    xaxis_title="Product Category",
    yaxis_title="Number of Products Sold",
    title_x=0.5,                       # centre the title
    bargap=0.20,                       # space between bars (optional)
)
fig.update_traces(marker_color='royalblue')  # Change bar color

fig.show()         


26/02/16 15:41:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+------------------+
|summary|  product_weight_g| product_length_cm| product_height_cm|  product_width_cm|
+-------+------------------+------------------+------------------+------------------+
|  count|             32949|             32949|             32949|             32949|
|   mean|2276.4724877841513| 30.81507784758263|16.937661234028347|23.196728277034204|
| stddev| 4282.038730977001|16.914458054065975|13.637554061749551|12.079047453227831|
|    min|                 0|                 7|                 2|                 6|
|    max|             40425|               105|               105|               118|
+-------+------------------+------------------+------------------+------------------+



26/02/16 15:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 15:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 15:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 15:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 15:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 15:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/16 1

+-------------------+-----+-----------------+-----------+
|product_category_en|count|          percent|cum_percent|
+-------------------+-----+-----------------+-----------+
|     BED_BATH_TABLE| 3029|9.192437255318502|       9.19|
|     SPORTS_LEISURE| 2867|8.700798154835969|      17.89|
|    FURNITURE_DECOR| 2657|8.063488209766016|      25.96|
|      HEALTH_BEAUTY| 2444|7.417073836909351|      33.37|
|         HOUSEWARES| 2335|7.086279627325423|      40.46|
+-------------------+-----+-----------------+-----------+
only showing top 5 rows


In [30]:

from pyspark.sql import SparkSession, functions as F


# cleaned = cleaned_dataframes
customers_info = cleaned["customers_info"]

#   Top 5 states with most sellers
df_customer_info =customers_info.groupby('customer_state').count().orderBy(F.desc("count"))
df_customer_info.withColumn('percent', (df_customer_info['count']/customers_info.count()) * 100).show(5)


df1_customer_info = df_customer_info.toPandas()


import plotly.graph_objects as go         
fig = go.Figure(
    data=[
        go.Pie( labels = df1_customer_info["customer_state"], 
                               values = df1_customer_info["count"], 
                               hole = 0.5 ,
                               pull = [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])
    ]
)
fig.update_layout(
    title = 'Olist customer over of all brazillian states (2016-2018)',
    autosize = False,
    width = 700,
    height = 500)
fig.update_traces(textposition = 'inside')
fig.update_layout(uniformtext_minsize = 12, uniformtext_mode = 'hide')

fig.show()


+--------------+-----+------------------+
|customer_state|count|           percent|
+--------------+-----+------------------+
|            SP|41746|41.980671956235355|
|            RJ|12852|12.924246538148248|
|            MG|11635|11.700405265433774|
|            RS| 5466| 5.496726702265665|
|            PR| 5045| 5.073360082863205|
+--------------+-----+------------------+
only showing top 5 rows


In [31]:
from pyspark.sql import SparkSession, functions as F

# assume you already have a SparkSession
spark = SparkSession.builder.getOrCreate()


# 1. Add weekday name & a numeric sort key
orders_week = (
    orders1
    .withColumn(
        "weekday_name",
        F.date_format("purchase_date", "EEEE")               # Monday, Tuesday…
    )
    .withColumn(
        "weekday_num",
        F.dayofweek("purchase_date")                         # 1=Sunday … 7=Saturday
    )
)

# 2. Aggregate by weekday
delivery_stats = (
    orders_week
    .groupBy("weekday_name", "weekday_num")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(
            F.when(F.col("delivered_date").isNotNull(), 1).otherwise(0)
        ).alias("delivered_orders")
    )
    # 3. Compute delivery rate
    .withColumn(
        "delivery_rate",
        F.col("delivered_orders") / F.col("total_orders")
    )
    # 4. Sort in calendar order
    .orderBy("weekday_num")
)

# 5. View the result
delivery_stats.show(truncate=False)


+------------+-----------+------------+----------------+------------------+
|weekday_name|weekday_num|total_orders|delivered_orders|delivery_rate     |
+------------+-----------+------------+----------------+------------------+
|Sunday      |1          |11960       |11634           |0.972742474916388 |
|Monday      |2          |16196       |15703           |0.9695603852803161|
|Tuesday     |3          |15963       |15502           |0.97112071665727  |
|Wednesday   |4          |15552       |15074           |0.9692644032921811|
|Thursday    |5          |14761       |14322           |0.970259467515751 |
|Friday      |6          |14122       |13685           |0.9690553745928339|
|Saturday    |7          |10887       |10556           |0.9695967667860751|
+------------+-----------+------------+----------------+------------------+



In [32]:
# Group by state and city
grouped_customers = customers_info.groupBy("customer_state", "customer_city").count()

# Add percent column to grouped data
grouped_customers = grouped_customers.withColumn(
    'percent', (F.col("count") / customers_info.count()) * 100).orderBy(F.desc("count"))


grouped_customers.show(5)

df1_customer_info = grouped_customers.toPandas()

fig=px.histogram(
    df1_customer_info,
    x="customer_city",
    y="count",
    nbins=100,
    title="Customer distribution – Olist (2016-2018)",

)
fig.update_layout(
    xaxis_title="Customer City",
    yaxis_title="Number of Customers",
    title_x=0.5,                       # centre the title
    bargap=0.20,                       # space between bars (optional)
)
fig.update_traces(marker_color='royalblue')  # Change bar color

fig.show()


+--------------+--------------+-----+------------------+
|customer_state| customer_city|count|           percent|
+--------------+--------------+-----+------------------+
|            SP|     Sao Paulo|15540|15.627356925211936|
|            RJ|Rio De Janeiro| 6882| 6.920686638308142|
|            MG|Belo Horizonte| 2773|2.7885882080831847|
|            DF|      Brasilia| 2131| 2.142979254030028|
|            PR|      Curitiba| 1521|1.5295501855371527|
+--------------+--------------+-----+------------------+
only showing top 5 rows


In [33]:
# print(order_reviews.count())
order_reviews = cleaned["order_reviews"]

# Only cast review_score to int if it looks like an integer, else set to None
order_reviews = order_reviews.withColumn(
    "review_score",
    F.when(F.col("review_score").rlike("^[0-9]+$"), F.col("review_score").cast("int"))
     .otherwise(None)
)
order_reviews.printSchema()

df_orders_reviews = (order_reviews           # use the cleaned DF
              .filter(F.col("review_score").isNotNull()) 
              .groupBy("review_score")
              .count()
              .withColumn("percent",
                          F.round(F.col("count") / total * 100, 2))
              .orderBy("review_score"))
df_orders_reviews.show()      # 1‑to‑5 ascending
            

df1_orders_reviews = df_orders_reviews.toPandas()

fig = px.bar(df1_orders_reviews, x="review_score", y="count", title = 'Order Reviews – Olist (2016-2018)')
fig.update_layout(
    xaxis_title="Review Score",
    yaxis_title="Number of Reviews",
    height=400,
    width=700,
    font=dict(  # Correctly specify the font property
        family="Arial, sans-serif",  # Font family
        size=14,                     # Font size
        color="black"                # Font color
    )
)
fig.show()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)

+------------+-----+-------+
|review_score|count|percent|
+------------+-----+-------+
|           0|    1|    0.0|
|           1|11424|  11.49|
|           2| 3151|   3.17|
|           3| 8179|   8.22|
|           4|19142|  19.25|
|           5|57328|  57.65|
+------------+-----+-------+



In [34]:
order_reviews = (
    order_reviews
    .join(orders.select("order_id"), "order_id")  # Keep only reviews with valid order_id in orders table
    .filter(F.col("review_score").isNotNull())   # Remove rows with null review scores
    .filter(F.col("review_creation_date").isNotNull())  # Remove rows without review creation date
)
order_reviews.show(5)


+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|            order_id|           review_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|c5a47daec61dfde19...|8a310b56e4d05a778...|           4|           Recomendo|  O pedido chegou a...| 2018-05-08 00:00:00|    2018-05-10 17:19:17|
|33dea0203a24cd38f...|bc3343847caad098a...|           2| Produto divergente |  Comprei por um ca...| 2018-05-16 00:00:00|    2018-05-17 13:47:25|
|dcbec508f4fc19506...|f51a5d398f2374cb2...|           1|                   6|  Eu não tenho notí...| 2018-05-02 00:00:00|    2018-05-02 11:17:26|
|2aaab7e991347226d...|2eab2c8f458335e0c...|           1|                Ruim|  Nao se vende e so...| 2018-08-12 00:00:00|   

In [35]:
order_payments= cleaned["order_payments"]
df_order_payments=order_payments.groupBy("payment_type").count().orderBy(F.desc("count"))
order_payments.show()
order_payments=order_payments.withColumn("payment_sequential", col("payment_sequential").cast("int"))\
      .withColumn("payment_installments", col("payment_installments").cast("int"))\
      .withColumn("payment_value", col("payment_value").cast("double"))
order_payments.printSchema()
order_payments.select(sum("payment_value")).show()
# order_payments.filter(F.col("payment_type").cast("string")=="not_defined").show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|53035288acc866ee1...|                 1| credit_card|                   1|       375.74|
|aa600269bffea363c...|                 1|      boleto|                   1|        93.26|
|caca64f71d7369571...|                 1| credit_card|                   1|       371.42|
|cccb62677b6ed93ec...|                 1| credit_card|                   7|        81.72|
|69796e4e03edbf9c0...|                 1|  debit_card|                   1|        93.22|
|6d0da5add693d6250...|                 1|      boleto|                   1|        19.48|
|e415a718c8f2f0372...|                 1| credit_card|                   2|        89.88|
|1dfc284468e80b96e...|                 1| credit_card|                   6|        78.97|
|ba254174e

In [36]:
order_payments.orderBy('payment_value',ascending=False).show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|03caa2c082116e1d3...|                 1| credit_card|                   1|     13664.08|
|736e1922ae60d0d6a...|                 1|      boleto|                   1|      7274.88|
|0812eb902a67711a1...|                 1| credit_card|                   8|      6929.31|
|fefacc66af859508b...|                 1|      boleto|                   1|      6922.21|
|f5136e38d1a14a4db...|                 1|      boleto|                   1|      6726.66|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows


In [37]:
pdf_order_payments = df_order_payments.toPandas()
colors = ["gold", "mediumturquoise", "darkorange", "lightgreen"]
fig = go.Figure(data=[go.Pie(
    labels=pdf_order_payments["payment_type"],
    values=pdf_order_payments["count"],
    hole=0.5,
    marker=dict(colors=colors,pattern=dict(shape=[".", "x", "+", "-"])),
    pull=[0.1, 0.1, 0.1, 0.1, 0.1,0.2,0.2,0.2]
)])
fig.update_layout(
    title='Shares of payment types of Olist customers (2016-2018)',
    autosize=False,
    width=600,
    height=500
)
fig.update_traces(textposition='inside')
fig.update_layout(uniformtext_minsize=12, uniformtext_mode='hide')

fig.show()

In [38]:

df_orders=(orders1.withColumn("duration_days",
      F.datediff("delivered_date", "purchase_date"))\
      .withColumn(
        "duration_hours",
       F.abs( F.expr("timestampdiff(HOUR, purchase_time, delivered_time)")))\
 .withColumn("delivery_delay_days",
            F.abs(F.datediff("delivered_date",
                             "estimated_date")))\
  .withColumn("is_weekend_delivery",
      dayofweek("delivered_date").isin(1, 7))  # Sunday=1, Saturday=7
)
df_orders.orderBy("purchase_date", ascending=True).show(5)
df_orders.printSchema()


+--------------------+--------------------+------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+-------------+--------------+-------------------+-------------------+
|            order_id|         customer_id|order_status|purchase_date|purchase_time|approved_date|approved_time|delivered_date|delivered_time|estimated_date|estimated_time|duration_days|duration_hours|delivery_delay_days|is_weekend_delivery|
+--------------------+--------------------+------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+-------------+--------------+-------------------+-------------------+
|2e7a8482f6fb09756...|08c5351a6aca1c158...|     Shipped|   2016-09-04|     21:15:19|   2016-10-07|     13:18:03|          NULL|          NULL|    2016-10-20|      00:00:00|         NULL|          NULL|               NULL|               NULL|
|e5fa5a7210941f7d5...|683c54fc24

In [39]:
from pyspark.sql import functions as F

# Assuming df_orders is your DataFrame


# Calculate average approval time in days by quarter
df_result = (
    df_orders
    .withColumn(
        "quarter_timestamp",
        F.concat_ws(" Q", F.year("purchase_date"), F.quarter("purchase_date"))
    )
    .withColumn(
        "approval_days",
        F.datediff("approved_date", "purchase_date")
    )
    .groupBy("quarter_timestamp")
    .agg(
        F.count("order_id").alias("no_purchases"),
        F.round(F.avg("approval_days"), 2).alias("avg_approval_days")
    )
    .orderBy("quarter_timestamp")
)

df_result.show(truncate=False)


+-----------------+------------+-----------------+
|quarter_timestamp|no_purchases|avg_approval_days|
+-----------------+------------+-----------------+
|2016 Q3          |4           |22.25            |
|2016 Q4          |325         |1.62             |
|2017 Q1          |5262        |0.43             |
|2017 Q2          |9349        |0.53             |
|2017 Q3          |12642       |0.52             |
|2017 Q4          |17848       |0.53             |
|2018 Q1          |21208       |0.48             |
|2018 Q2          |19979       |0.53             |
|2018 Q3          |12820       |0.53             |
|2018 Q4          |4           |NULL             |
+-----------------+------------+-----------------+



In [40]:

geolocation = cleaned["geolocation"]
import unicodedata
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Step 3: Standardize city names (title case) and trim
def normalize_accents(text):
    if text:
        return unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    return text

normalize_udf = udf(normalize_accents, StringType())
geo = geolocation.withColumn("geolocation_city", F.lower(F.trim(F.col("geolocation_city")))) \
         .withColumn("geolocation_city", normalize_udf(F.col("geolocation_city"))) \
         .withColumn("geolocation_city", F.initcap(F.col("geolocation_city")))
# Step 5: Group by zip, take average of lat/lng, first city/state
geo= geo.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("latitude"),
    F.avg("geolocation_lng").alias("longitude"),
    F.first("geolocation_city").alias("city"),
    F.first("geolocation_state").alias("state")
)
geo.show(5, truncate=False)

+---------------------------+-------------------+-------------------+---------+-----+
|geolocation_zip_code_prefix|latitude           |longitude          |city     |state|
+---------------------------+-------------------+-------------------+---------+-----+
|01022                      |-23.544780826808893|-46.632062988389954|Sao Paulo|SP   |
|01029                      |-23.540574380051   |-46.63312870430693 |Sao Paulo|SP   |
|01033                      |-23.53914866486542 |-46.63602255146723 |Sao Paulo|SP   |
|01050                      |-23.54918231337722 |-46.64309316226118 |Sao Paulo|SP   |
|01124                      |-23.528361038348763|-46.63330445274645 |Sao Paulo|SP   |
+---------------------------+-------------------+-------------------+---------+-----+
only showing top 5 rows


In [41]:
# print(df_orders.columns)

df=(df_orders.alias("o").join(order_items.alias("oi"),on="order_id",how="inner"))
df=(df.alias("d").join(sellers.alias("s"),on="seller_id",how="left"))
df=(df.alias("d1").join(products.alias("p"),on="product_id",how="left"))
df=(df.alias("d2").join(customers_info.alias("ci"),on="customer_id",how="inner"))
df=(df.alias("d4").join(order_reviews.alias("or"),on="order_id",how="left"))
df=(df.alias("d5").join(order_payments.alias("op"),on="order_id",how="left"))
df=(df.alias("d6").join(geo.alias("g"),customers_info.customer_zip_code_prefix == geo.geolocation_zip_code_prefix,how="left"))
df = df.drop('order_approved_at', 'order_delivered_carrier_date', 'seller_zip_code_prefix', 'product_name_lenght', 'product_description_lenght',
        'product_photos_qty', 'customer_zip_code_prefix', 'review_id', 'review_comment_title', 'review_comment_message',
        'review_creation_date', 'review_answer_timestamp',"customer_unique_id","order_item_id"
       )
# product_category_en is null Fill with 'Unknown' or 'Other'
df = df.fillna({'product_category_en': 'Unknown'})

# Recommended: mark as -1 to differentiate from actual score 0
df = df.fillna({"review_score": -1})

# Ensure the columns are cast to numeric types before calculating the median


for col_name in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    # Calculate the median
    median_vals = df.approxQuantile(col_name, [0.5], 0.05)
    if median_vals:
        median_val = median_vals[0]
        df = df.fillna({col_name: median_val})  # Fill nulls with the median
  
from pyspark.sql.functions import year, quarter, col, count, sum as _sum, first


# df.orderBy("purchase_date", ascending=True).show(100)

# # Remove rows where both latitude and longitude are NULL
df = df.filter(
    (col("geolocation_zip_code_prefix").isNotNull()) &
    (col("latitude").isNotNull()) &
    (col("longitude").isNotNull()) &
    (col("city").isNotNull()) &
    (col("state").isNotNull()) &
    (col("payment_sequential").isNotNull()) &
    ((col("payment_installments").cast("int")) > 0)  # Ensure this is a boolean condition
)

df = df.withColumn("latitude", F.round(col("latitude"), 5)) \
       .withColumn("longitude", F.round(col("longitude"), 5))

# Step 2: Group by everything else and aggregate
df_agg = df.groupBy("order_id").agg(
    _sum("price").alias("Total_price"),
    _sum("freight_value").alias("Total_freight_value")
)
df = df.drop("price", "freight_value") \
            .dropDuplicates(["order_id"])


df= df.join(df_agg, on="order_id", how="left")


# df.select(sum("total_payment")).show()
df.orderBy("purchase_date", ascending=True).show(1)
print(df.count())


+--------------------+--------------------+--------------------+--------------------+------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+-------------+--------------+-------------------+-------------------+-----------+-------------+-------------+-----------+------------+----------------+-----------------+-----------------+----------------+-------------------+-------------+--------------+------------+------------------+------------+--------------------+-------------+---------------------------+--------+---------+---------+-----+-----------+-------------------+
|            order_id|         customer_id|          product_id|           seller_id|order_status|purchase_date|purchase_time|approved_date|approved_time|delivered_date|delivered_time|estimated_date|estimated_time|duration_days|duration_hours|delivery_delay_days|is_weekend_delivery|gross_value|shipping_date|shipping_time|seller_city|seller_state|product_w

98389


In [42]:


# Group products by category and sum payment_value to get total revenue per category
df_revenue = df.groupBy("product_category_en").agg(
    sum("payment_value").alias("total_revenue")
).orderBy("total_revenue", ascending=False)

# Show only the top 10 categories to avoid memory issues
df_revenue.show(10, truncate=False)


+---------------------+------------------+
|product_category_en  |total_revenue     |
+---------------------+------------------+
|HEALTH_BEAUTY        |1420003.0299999993|
|WATCHES_GIFTS        |1279029.9000000004|
|BED_BATH_TABLE       |1211427.6899999997|
|SPORTS_LEISURE       |1140475.0200000005|
|COMPUTERS_ACCESSORIES|1040222.9199999997|
|FURNITURE_DECOR      |888657.9899999995 |
|HOUSEWARES           |765530.3400000002 |
|COOL_STUFF           |703616.2399999998 |
|AUTO                 |672385.71         |
|GARDEN_TOOLS         |567982.5900000001 |
+---------------------+------------------+
only showing top 10 rows


In [43]:
# Use the correct products DataFrame from cleaned

orders1.show()
orders1.select([sum(col(c).isNull().cast("int")).alias(c) for c in orders1.columns]).show()
order_items.show()
order_items.select([sum(col(c).isNull().cast("int")).alias(c) for c in order_items.columns]).show()
for col_name in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    # Calculate the median
    median_vals = products.approxQuantile(col_name, [0.5], 0.05)
    if median_vals:
        median_val = median_vals[0]
        products = products.fillna({col_name: median_val})  # Fill nulls with the median
products=products.drop('product_name_lenght', 'product_description_lenght',
        'product_photos_qty')       
products = products.fillna({'product_category_en': 'Unknown'})
products.show()
products.select([sum(col(c).isNull().cast("int")).alias(c) for c in products.columns]).show()
sellers.show()
customers_info.show()
order_reviews=order_reviews.drop('review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp')
order_reviews.show()
order_reviews.select([sum(col(c).isNull().cast("int")).alias(c) for c in order_reviews.columns]).show()
order_payments.show()
order_payments.select([sum(col(c).isNull().cast("int")).alias(c) for c in order_payments.columns]).show()
geo.show()
geo.select([sum(col(c).isNull().cast("int")).alias(c) for c in geo.columns]).show()


+--------------------+--------------------+------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+
|            order_id|         customer_id|order_status|purchase_date|purchase_time|approved_date|approved_time|delivered_date|delivered_time|estimated_date|estimated_time|
+--------------------+--------------------+------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+
|974c1993ab8024d3e...|a90391a47de936d56...|   Delivered|   2017-02-20|     11:45:39|   2017-02-22|     03:10:20|    2017-03-09|      14:27:58|    2017-03-21|      00:00:00|
|c82430e4a84f03b67...|a625f7cf467b00ae8...|   Delivered|   2017-11-03|     12:56:49|   2017-11-07|     07:30:43|    2017-11-21|      18:23:02|    2017-11-29|      00:00:00|
|4630b9ea86d473a93...|5b6e97fea8528cf00...|   Delivered|   2018-06-27|     16:11:38|   2018-06-27|     16:29:31|    2018-07-03|      04

+---------------------------+-------------------+-------------------+---------+-----+
|geolocation_zip_code_prefix|           latitude|          longitude|     city|state|
+---------------------------+-------------------+-------------------+---------+-----+
|                      01022|-23.544780826808893|-46.632062988389954|Sao Paulo|   SP|
|                      01029|   -23.540574380051| -46.63312870430693|Sao Paulo|   SP|
|                      01033| -23.53914866486542| -46.63602255146723|Sao Paulo|   SP|
|                      01050| -23.54918231337722| -46.64309316226118|Sao Paulo|   SP|
|                      01124|-23.528361038348763| -46.63330445274645|Sao Paulo|   SP|
|                      01150| -23.53278643657309|-46.658459923150886|Sao Paulo|   SP|
|                      01189| -23.53210825506378| -46.63839470107914|Sao Paulo|   SP|
|                      01201|-23.536094768181357| -46.64976590706883|Sao Paulo|   SP|
|                      01205| -23.53546316966787|-46.6

In [44]:
df.write.mode("overwrite").csv("path/to/output_folder/output.csv")

In [46]:
# Add this at the top of your notebook, before any writes or actions
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [47]:
df.write.mode("overwrite").parquet("final_table.parquet")

orders1.write.mode("overwrite").parquet("orders_table.parquet")
products.write.mode("overwrite").parquet("products_table.parquet")
customers_info.write.mode("overwrite").parquet("customers_table.parquet")
order_items.write.mode("overwrite").parquet("order_items_table.parquet")
order_payments.write.mode("overwrite").parquet("order_payments_table.parquet")
order_reviews.write.mode("overwrite").parquet("order_reviews_table.parquet")
sellers.write.mode("overwrite").parquet("sellers_table.parquet")
geo.write.mode("overwrite").parquet("geographies_table.parquet")



26/02/16 17:41:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/16 17:41:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/16 17:41:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/02/16 17:41:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/02/16 17:41:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/16 17:41:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/16 17:41:53 WARN MemoryManager: Total allocation exceeds 95.00%